# BusBooking — nghiệm thu kiến trúc độc lập trên Kaggle
Upload toàn bộ `load-tests/` thành một Dataset và JAR Spring Boot thực thi thành Dataset riêng. Add Input cả hai. Accelerator=None, Internet=On. Run All trên kernel mới.
Bộ chạy không sửa JAR, migration, pool, rate limit hoặc logic nghiệp vụ. Nó tạo PostgreSQL/Mailpit riêng, ghi dữ liệu test và gây lỗi có kiểm soát trên các tiến trình do nó sở hữu.
Mặc định MODE='smoke': chạy nhanh tuần tự auth/OTP/login → catalog/booking/hủy → read/write/mixed mỗi lượt 1 VU → đối soát → ZIP. Sau khi SMOKE_PASS, đổi MODE='acceptance' và chạy một lượt mới để nghiệm thu đầy đủ, có thể mất nhiều giờ.
Java, DB và Locust dùng chung Kaggle: chỉ xác nhận giới hạn môi trường này; cần máy tạo tải riêng để khẳng định công suất backend độc lập. Đọc `load-tests/ACCEPTANCE.md` trước khi chạy.

In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, shutil, subprocess, sys, tempfile, zipfile

SOURCE_OVERRIDE = ''  # Absolute load-tests root if multiple datasets contain it
JAR_OVERRIDE = ''     # Absolute executable JAR path if multiple candidates exist
MODE = 'smoke'        # 'smoke' first; change to 'acceptance' for the full suite
# Optional test-only settings; changing these creates a different experiment.
OVERRIDES = {}        # Optional explicit harness parameter overrides

roots = [p.parent for p in Path('/kaggle/input').rglob('requirements-kaggle.txt')
         if (p.parent / 'scripts/run_acceptance.py').is_file()]
WORK = Path('/kaggle/working')
kit_roots = []
if not roots and not SOURCE_OVERRIDE:
    for packed in Path('/kaggle/input').rglob('busbooking-kaggle-kit.zip'):
        target = Path(tempfile.mkdtemp(prefix='bb-kit-', dir=WORK))
        with zipfile.ZipFile(packed) as bundle:
            manifest = json.loads(bundle.read('kit-manifest.json'))
            assert set(bundle.namelist()) == set(manifest) | {'kit-manifest.json'}, 'Kit inventory mismatch'
            for name, expected_sha in manifest.items():
                output = (target / name).resolve()
                assert output.is_relative_to(target.resolve()), 'Unsafe kit path'
                output.parent.mkdir(parents=True, exist_ok=True)
                with bundle.open(name) as source, output.open('wb') as dest: shutil.copyfileobj(source, dest)
                with output.open('rb') as data: assert hashlib.file_digest(data, 'sha256').hexdigest() == expected_sha, 'Kit digest mismatch'
        roots.append(target / 'load-tests')
        kit_roots.append(target)
if SOURCE_OVERRIDE:
    SOURCE = Path(SOURCE_OVERRIDE).resolve()
else:
    assert len(roots) == 1, f'Choose SOURCE_OVERRIDE; candidates: {roots}'
    SOURCE = roots[0].resolve()
jars = []
for path in [p for tree in [Path('/kaggle/input'), *kit_roots] for p in tree.rglob('*.jar')]:
    try:
        with zipfile.ZipFile(path) as z:
            if 'BOOT-INF/classes/com/example/busbooking/BusBookingApplication.class' in z.namelist():
                jars.append(path)
    except zipfile.BadZipFile:
        pass
if JAR_OVERRIDE:
    JAR = Path(JAR_OVERRIDE).resolve()
else:
    assert len(jars) == 1, f'Choose JAR_OVERRIDE; candidates: {jars}'
    JAR = jars[0].resolve()
assert (SOURCE / 'config/acceptance.yaml').is_file() and JAR.is_file()
WORK = Path('/kaggle/working')
assert MODE in ('smoke', 'acceptance')
RUN = WORK / (MODE + '-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S') + '-' + str(os.getpid()))
print('Mode:', MODE, '\nHarness:', SOURCE, '\nJAR:', JAR, '\nOutput:', RUN)

## Cài đúng phiên bản trong môi trường riêng
Tạo venv dưới /kaggle/working, không thay package của kernel. Suite chạy bằng Python mới để tránh giữ phiên bản psutil cũ.

In [ ]:
VENV = WORK / (RUN.name + '-venv')
assert (SOURCE / 'scripts/bootstrap_environment.py').is_file(), 'Dataset kit cũ: cập nhật kit mới trước khi chạy notebook này'
subprocess.run([sys.executable, '-B', str(SOURCE / 'scripts/bootstrap_environment.py'), '--venv', str(VENV), '--requirements', str(SOURCE / 'requirements-kaggle.txt')], check=True)
PYTHON = VENV / 'bin/python'
env = {k:v for k,v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV')}
env.update(PYTHONDONTWRITEBYTECODE='1', PYTHONUNBUFFERED='1', PYTHONNOUSERSITE='1', BB_VALIDATION_DIR=str(WORK / 'harness-validation'))
subprocess.run([str(PYTHON), '-c', "from importlib.metadata import version; import wrapt; print({n:version(n) for n in ['locust','psutil','psycopg','requests','PyYAML','bcrypt','wrapt']})"], env=env, check=True)
# Write only an explicit experiment config under /kaggle/working, not application properties.
config_path = WORK / (RUN.name + '-config.yaml')
subprocess.run([str(PYTHON), '-B', str(SOURCE / 'scripts/configure_run.py'), '--mode', MODE, '--output', str(config_path), '--overrides', json.dumps(OVERRIDES)], env=env, check=True)
# Full harness self-tests run before acceptance; smoke remains the minimal system flow.
if MODE == 'acceptance':
    subprocess.run([str(PYTHON), '-B', '-m', 'unittest', 'discover', '-s', str(SOURCE / 'tests'), '-v'], env=env, check=True)

## Chạy theo MODE đã chọn
Không chạy lặp cell này với cùng RUN. Smoke thành công có SMOKE_PASS và exit 0, chỉ xác nhận luồng kiểm thử. Acceptance mới chạy đầy đủ. Exit 2 là FAIL/UNKNOWN/thiếu bằng chứng; vẫn xuất báo cáo và ZIP.

In [ ]:
assert not RUN.exists(), 'Use a fresh run directory; previous evidence is immutable'
command = [str(PYTHON), '-B', str(SOURCE / 'scripts/run_acceptance.py'), '--jar', str(JAR), '--config', str(config_path), '--output', str(RUN)]
process = subprocess.Popen(command, env=env)
try:
    exit_code = process.wait()
except KeyboardInterrupt:
    process.terminate()
    try: exit_code = process.wait(timeout=120)
    except subprocess.TimeoutExpired:
        print('Cleanup still pending; inspect owned-process logs before another run')
        exit_code = None
print('Suite exit code:', exit_code)
subprocess.run([str(PYTHON), '-B', str(SOURCE / 'scripts/export_results.py'), str(RUN)], env=env, check=True)
assert (RUN / 'REPORT.md').exists(), 'No report; inspect startup/kernel logs'

In [ ]:
from IPython.display import display, Markdown, FileLink, SVG
display(Markdown((RUN / 'REPORT.md').read_text(encoding='utf-8')))
display(SVG(filename=str(RUN / 'throughput.svg')))
display(FileLink(str(RUN.with_suffix('.zip'))))
print('Detailed status:', json.loads((RUN / 'summary.json').read_text()))